# Drone Object Detection — Subset Comparison Training

Run all cells top to bottom. This notebook trains **four YOLOv8 variants** on a random
subset of the training data (controlled by `SUBSET_FRACTION`) using fewer epochs — the
goal is fast model selection, not final performance.

**Workflow:**
1. Run this notebook to compare all four architectures cheaply
2. Open `colab/results/analyze.ipynb` to review mAP50 results
3. Pick the best model, set `BEST_MODEL` in `colab/train_full.ipynb`, and run full training

| Model | Base Architecture | Notes |
|-------|------------------|-------|
| `yolov8n` | YOLOv8 Nano | Lightweight baseline (~3.2 M params) |
| `yolov8s` | YOLOv8 Small | Intermediate capacity (~11.2 M params) |
| `yolov8m` | YOLOv8 Medium | High capacity (~25.9 M params) |
| `yolov8n_tuned` | YOLOv8 Nano | Nano + augmentation hyperparameter tuning |

In [ ]:
# -- Cell 1: Configuration ---------------------------------------------------
# Edit DRIVE_BASE if your Drive path differs.
# SUBSET_FRACTION controls what fraction of training images are used for comparison.
# SEED ensures the same subset is sampled every run (reproducible).

DRIVE_BASE       = '/content/drive/MyDrive/drone_detection_training'
DATASET_ZIP      = 'combined_yolo.zip'
SUBSET_FRACTION  = 0.35   # ~35% of train images (~133 of 378)
SEED             = 42

MODELS_TO_RUN = [
    {
        'name':    'yolov8n',
        'weights': 'yolov8n.pt',
        'epochs':  20,
        'imgsz':   640,
        'batch':   16,
    },
    {
        'name':    'yolov8s',
        'weights': 'yolov8s.pt',
        'epochs':  20,
        'imgsz':   640,
        'batch':   12,
    },
    {
        'name':    'yolov8m',
        'weights': 'yolov8m.pt',
        'epochs':  20,
        'imgsz':   640,
        'batch':   8,
    },
    {
        'name':    'yolov8n_tuned',
        'weights': 'yolov8n.pt',
        'epochs':  20,
        'imgsz':   640,
        'batch':   16,
        'lr0':     0.001,
        'lrf':     0.01,
        'mosaic':  1.0,
        'hsv_h':   0.015,
        'hsv_s':   0.7,
        'hsv_v':   0.4,
        'flipud':  0.1,
        'fliplr':  0.5,
    },
]

print('Models queued:   ', [m['name'] for m in MODELS_TO_RUN])
print('Drive base:      ', DRIVE_BASE)
print('Dataset zip:     ', DATASET_ZIP)
print(f'Subset fraction:  {SUBSET_FRACTION} (seed={SEED})')

Models queued:    ['yolov8n', 'yolov8s', 'yolov8m', 'yolov8n_tuned']
Drive base:       /content/drive/MyDrive/drone_detection_training
Dataset zip:      combined_yolo.zip
Subset fraction:  0.35 (seed=42)


## Performance Metric: mAP50

**Why mAP50?**

Mean Average Precision at IoU >= 0.50 is the standard metric for object detection
benchmarks (COCO, VisDrone) and the correct choice for this project:

- **Precision + recall together**: mAP summarizes the precision-recall curve at all
  confidence thresholds, averaged across classes. Single-number metrics like accuracy
  collapse this tradeoff.
- **Class imbalance**: The dataset is skewed (vehicle 54%, person 35%, two-wheeler 10%).
  Accuracy would be dominated by the majority class. mAP weights each class equally
  via per-class AP averaging.
- **Small objects**: Drone imagery contains many small objects. IoU = 0.50 is appropriate;
  stricter thresholds penalize minor localization errors irrelevant at flight altitude.
- **mAP50-95**: Also tracked as a secondary metric (average over IoU 0.50-0.95),
  consistent with the COCO benchmark.

Per-class AP (person, vehicle, two-wheeler) is available in `results.csv` to detect
class-specific underperformance.

## Cross-Validation Strategy

**Approach: three-way train / validation / test split**

True k-fold cross-validation is prohibitively expensive here (50 epochs x 4 models x k folds
= hundreds of GPU-hours). We use the standard COCO/YOLO approach instead:

| Split | Purpose | Approximate Size |
|-------|---------|------------------|
| Train | Model weight fitting | ~70% of data |
| Validation | Hyperparameter selection, early-stopping monitor | ~15% |
| Test | **Held-out** final evaluation only | ~15% |

**Why this is valid:**
- Splits are **scene-level** from the original VisDrone and UAVDT datasets — no
  frame-level leakage between train/val/test sequences.
- The test set is untouched during all model selection; only used for final evaluation.
- All four model variants are evaluated on the **same** val and test sets, ensuring
  a fair comparison.
- This is standard practice for COCO-style detection benchmarks (YOLOv5/v8, Detectron2).

In [ ]:
# -- Cell 2: Install Dependencies --------------------------------------------
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'ultralytics>=8.0.0', 'torch', 'torchvision'],
    check=True
)

import torch

if torch.cuda.is_available():
    device = torch.cuda.get_device_name(0)
    mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'GPU: {device}  |  VRAM: {mem_gb:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be very slow on CPU.')
    print('Go to Runtime -> Change runtime type -> GPU')

GPU: NVIDIA A100-SXM4-40GB  |  VRAM: 39.5 GB


In [ ]:
# -- Cell 3: Mount Google Drive ----------------------------------------------
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

base         = Path(DRIVE_BASE)
runs_dir     = base / 'runs'
datasets_dir = base / 'datasets'

runs_dir.mkdir(parents=True, exist_ok=True)
datasets_dir.mkdir(parents=True, exist_ok=True)

print('Drive folder structure:')
print(f'  {base}/')
print(f'  runs/      {"(exists)" if runs_dir.exists() else "(created)"}')
print(f'  datasets/  {"(exists)" if datasets_dir.exists() else "(created)"}')

Mounted at /content/drive
Drive folder structure:
  /content/drive/MyDrive/drone_detection_training/
  runs/      (exists)
  datasets/  (exists)


In [ ]:
# -- Cell 4: Dataset Integrity Check -----------------------------------------
import shutil
from pathlib import Path

DATASET_DIR = Path('/content/dataset')
zip_path    = Path(DRIVE_BASE) / 'datasets' / DATASET_ZIP

if not zip_path.exists():
    print('ERROR: Dataset zip not found.')
    print()
    print('Please upload the zip to Google Drive at:')
    print(f'  {zip_path}')
    print()
    print('To create the zip locally, run:')
    print('  python scripts/zip_for_colab.py --dataset combined')
    print('Then upload colab_uploads/' + DATASET_ZIP + ' to your Drive at the path above.')
    raise SystemExit(1)

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)

print(f'Unzipping {DATASET_ZIP} to {DATASET_DIR} ...')
shutil.unpack_archive(str(zip_path), str(DATASET_DIR))
print('Unzip complete.')

errors = {}
counts = {}

for split in ('train', 'val', 'test'):
    img_dir = DATASET_DIR / 'images' / split
    lbl_dir = DATASET_DIR / 'labels' / split

    if not img_dir.exists():
        errors[split] = f'Missing: images/{split}/'
        continue
    if not lbl_dir.exists():
        errors[split] = f'Missing: labels/{split}/'
        continue

    imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.jpeg')) + list(img_dir.glob('*.png'))
    lbls = list(lbl_dir.glob('*.txt'))
    counts[split] = {'images': len(imgs), 'labels': len(lbls)}

    if len(imgs) != len(lbls):
        errors[split] = f'{split}: {len(imgs)} images vs {len(lbls)} labels (mismatch)'

yaml_files = list(DATASET_DIR.glob('*.yaml'))
if not yaml_files:
    errors['yaml'] = 'No .yaml file found in dataset root'
else:
    yaml_file = yaml_files[0]

print()
print(f'{"Split":<8} {"Images":>8} {"Labels":>8} {"Match":>7}')
print('-' * 35)
for split, c in counts.items():
    match = 'OK' if c['images'] == c['labels'] else 'MISMATCH'
    print(f'{split:<8} {c["images"]:>8} {c["labels"]:>8} {match:>7}')

if yaml_files:
    print(f'\nDataset yaml: {yaml_file.name}')

if errors:
    print()
    print('ERRORS:')
    for e in errors.values():
        print(f'  - {e}')
    raise SystemExit(1)

print('\nDataset integrity check passed.')

Unzipping combined_yolo.zip to /content/dataset ...
Unzip complete.

Split      Images   Labels   Match
-----------------------------------
train        7737     7737      OK
val           819      819      OK
test         1882     1882      OK

Dataset yaml: combined.yaml

Dataset integrity check passed.


In [ ]:
# -- Cell 5: Build Subset Dataset --------------------------------------------
# Samples SUBSET_FRACTION of training images and writes a YAML that points
# training to that subset while keeping val/test on the full original splits.

import random
import shutil
import yaml
from pathlib import Path

random.seed(SEED)

DATASET_DIR = Path('/content/dataset')
SUBSET_DIR  = Path('/content/subset')

# Collect all training images
all_train_imgs = sorted(
    list((DATASET_DIR / 'images' / 'train').glob('*.jpg')) +
    list((DATASET_DIR / 'images' / 'train').glob('*.jpeg')) +
    list((DATASET_DIR / 'images' / 'train').glob('*.png'))
)
n_total  = len(all_train_imgs)
n_subset = max(1, int(n_total * SUBSET_FRACTION))
subset_imgs = random.sample(all_train_imgs, n_subset)

# Build subset image/label directories
subset_img_dir = SUBSET_DIR / 'images' / 'train'
subset_lbl_dir = SUBSET_DIR / 'labels' / 'train'
if SUBSET_DIR.exists():
    shutil.rmtree(SUBSET_DIR)
subset_img_dir.mkdir(parents=True)
subset_lbl_dir.mkdir(parents=True)

missing_labels = 0
for img_path in subset_imgs:
    lbl_path = DATASET_DIR / 'labels' / 'train' / (img_path.stem + '.txt')
    shutil.copy2(img_path, subset_img_dir / img_path.name)
    if lbl_path.exists():
        shutil.copy2(lbl_path, subset_lbl_dir / lbl_path.name)
    else:
        missing_labels += 1

# Read original YAML to get nc and names
orig_yaml_files = list(DATASET_DIR.glob('*.yaml'))
with open(orig_yaml_files[0]) as fh:
    orig_cfg = yaml.safe_load(fh)

# Write subset YAML with absolute paths so val/test point to original splits
subset_yaml_data = {
    'path':  '/content',
    'train': 'subset/images/train',
    'val':   str(DATASET_DIR / 'images' / 'val'),
    'test':  str(DATASET_DIR / 'images' / 'test'),
    'nc':    orig_cfg['nc'],
    'names': orig_cfg['names'],
}
SUBSET_YAML_PATH = '/content/subset.yaml'
with open(SUBSET_YAML_PATH, 'w') as fh:
    yaml.dump(subset_yaml_data, fh, default_flow_style=False)

print(f'Subset training set: {n_subset} / {n_total} images ({SUBSET_FRACTION:.0%})')
if missing_labels:
    print(f'  WARNING: {missing_labels} images had no matching label file')
print(f'Val / test:  full original splits (unchanged)')
print(f'Subset YAML: {SUBSET_YAML_PATH}')

Subset training set: 2707 / 7737 images (35%)
Val / test:  full original splits (unchanged)
Subset YAML: /content/subset.yaml


In [ ]:
# -- Cell 6: Automated Training Loop -----------------------------------------
import json
import time
import shutil
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

yaml_path = SUBSET_YAML_PATH  # defined in the subset-build cell above
print(f'Dataset yaml: {yaml_path}')
print()

training_summary = []

for cfg in MODELS_TO_RUN:
    model_name = cfg['name']
    drive_out  = Path(DRIVE_BASE) / 'runs' / model_name

    # Skip if already completed (idempotent across Colab sessions)
    if (drive_out / 'results.csv').exists():
        print(f'[SKIP] {model_name} -- results already on Drive')
        training_summary.append({'name': model_name, 'status': 'skipped'})
        continue

    sep = '=' * 60
    print(f'\n{sep}')
    print(f'  Training: {model_name}')
    print(sep)

    model        = YOLO(cfg['weights'])
    train_kwargs = {k: v for k, v in cfg.items() if k not in ('name', 'weights')}

    t_start = time.time()
    model.train(
        data=yaml_path,
        project='/content/runs',
        name=model_name,
        exist_ok=False,
        **train_kwargs,
    )
    training_time = time.time() - t_start

    # Build run_info.json
    run_dir       = Path('/content/runs') / model_name
    best_pt       = run_dir / 'weights' / 'best.pt'
    model_size_mb = round(best_pt.stat().st_size / 1e6, 2) if best_pt.exists() else None

    df = pd.read_csv(run_dir / 'results.csv')
    df.columns = df.columns.str.strip()
    best_idx = df['metrics/mAP50(B)'].idxmax()

    run_info = {
        'model_name':            model_name,
        'weights':               cfg['weights'],
        'dataset':               f'subset ({SUBSET_FRACTION:.0%} of train)',
        'epochs_completed':      len(df),
        'best_epoch':            int(best_idx) + 1,
        'training_time_seconds': round(training_time, 1),
        'model_size_mb':         model_size_mb,
        'config':                {k: v for k, v in cfg.items() if k not in ('name', 'weights')},
        'best_metrics': {
            'mAP50':     round(float(df.loc[best_idx, 'metrics/mAP50(B)']),    4),
            'mAP50_95':  round(float(df.loc[best_idx, 'metrics/mAP50-95(B)']), 4),
            'precision': round(float(df.loc[best_idx, 'metrics/precision(B)']), 4),
            'recall':    round(float(df.loc[best_idx, 'metrics/recall(B)']),   4),
        },
    }
    with open(run_dir / 'run_info.json', 'w') as fh:
        json.dump(run_info, fh, indent=2)

    # Atomic copy to Drive
    tmp_dst = drive_out.parent / f'{model_name}_tmp'
    if tmp_dst.exists():
        shutil.rmtree(tmp_dst)

    (tmp_dst / 'weights').mkdir(parents=True, exist_ok=True)
    for wf in (run_dir / 'weights').glob('*.pt'):
        shutil.copy2(str(wf), str(tmp_dst / 'weights' / wf.name))

    for item in run_dir.rglob('*'):
        if item.is_file() and 'weights' not in item.parts:
            rel    = item.relative_to(run_dir)
            target = tmp_dst / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(item), str(target))

    if drive_out.exists():
        shutil.rmtree(drive_out)
    tmp_dst.rename(drive_out)

    n_saved = sum(1 for _ in drive_out.rglob('*') if _.is_file())
    bm      = run_info['best_metrics']
    m50     = bm['mAP50']
    m5095   = bm['mAP50_95']
    t_min   = training_time / 60
    print(f'[DONE] {model_name}')
    print(f'       mAP50={m50}  mAP50-95={m5095}  time={t_min:.1f}min  size={model_size_mb}MB  files={n_saved}')

    training_summary.append({'name': model_name, 'status': 'trained', **bm})

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset yaml: /content/subset.yaml


  Training: yolov8n
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/subset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, 

In [ ]:
# -- Cell 7: Session Summary + Export Results --------------------------------
import json
import pandas as pd
from pathlib import Path

summary = globals().get('training_summary', [])
if not summary:
    print('No training data. Run the training loop cell first.')
else:
    rows = []
    for s in summary:
        rows.append({
            'model':     s['name'],
            'status':    s['status'].upper(),
            'mAP50':     s.get('mAP50'),
            'mAP50-95':  s.get('mAP50_95'),
            'precision': s.get('precision'),
            'recall':    s.get('recall'),
        })
    df = pd.DataFrame(rows).set_index('model')
    print('Subset Comparison Summary')
    print('=' * 60)
    print(df.to_string())

    # Save a consolidated JSON for analyze.ipynb to consume
    summary_path = Path(DRIVE_BASE) / 'subset_results_summary.json'
    with open(summary_path, 'w') as fh:
        json.dump(summary, fh, indent=2)
    print(f'\nResults JSON saved to: {summary_path}')

    trained = [s for s in summary if s['status'] == 'trained']
    if trained:
        best = max(trained, key=lambda s: s.get('mAP50', 0))
        print(f'\nTop model by mAP50: {best["name"]}  ({best.get("mAP50")})')

    print()
    print('Next steps:')
    print('  1. Download each model folder from Drive:')
    print(f'     {DRIVE_BASE}/runs/<model_name>/')
    print('  2. Place each folder into: colab/results/<model_name>/')
    print('  3. Open colab/results/analyze.ipynb and run all cells to compare.')
    print('  4. Set BEST_MODEL in colab/train_full.ipynb and run full training.')

Subset Comparison Summary
                status   mAP50  mAP50-95  precision  recall
model                                                      
yolov8n        TRAINED  0.4365    0.2068     0.5498  0.4389
yolov8s        TRAINED  0.5214    0.2594     0.6411  0.5002
yolov8m        TRAINED  0.5663    0.2899     0.6770  0.5404
yolov8n_tuned  TRAINED  0.4311    0.2051     0.5359  0.4363

Results JSON saved to: /content/drive/MyDrive/drone_detection_training/subset_results_summary.json

Top model by mAP50: yolov8m  (0.5663)

Next steps:
  1. Download each model folder from Drive:
     /content/drive/MyDrive/drone_detection_training/runs/<model_name>/
  2. Place each folder into: colab/results/<model_name>/
  3. Open colab/results/analyze.ipynb and run all cells to compare.
  4. Set BEST_MODEL in colab/train_full.ipynb and run full training.
